# **Data Analysis II**
## *Lab 4: Cut-based vs Machine Learning*

In the fortunate case where we had the $Z → μ^+μ^-$ process, which has very little background contamination, cut-based analyses are a quick way to separate your signal from the background processes. But what happens when we have a more challenging case? Let's take a look at the following and try to find some improvements.

In [ ]:
import atlasopenmagic as atom
import awkward as ak
import uproot
import numpy as np
import scipy as sp
import matplotlib.pyplot as plt
import vector
import os

### $H → ZZ^* → 4\ell$

We will step away from our 2 muon final state and look at the Higgs boson decay into 4 leptons. This is one of the decays that was thoroughly studied for the discovery of the Higgs boson back in 2012, along with $H → γγ$. These 2 processes have the advantage of being very clean and their final states being easy to reconstruct.

The main backgrounds that mimic our signal are:
 - $ZZ^{(*)}$ production: either on- or off-shell, processes involving Z boson pair production have the same final state as $H → 4ℓ$, but at very different invariant masses.
 - $t\bar{t}$ or single-$t$ production: these can yield multiple leptons in the final state, at masses comparable with $m_H$.
 - $VVV$ production: depending on the channel.

In [ ]:
atom.set_release('2025e-13tev-beta')

defs = {
    r'Data':{'dids':['data']},
    r'Background $Z,t\bar{t},t\bar{t}+V,VVV$':{'dids': [364245, 364247, 410408,
                                                        410470, 410155, 410218,
                                                        410219, 410220, 412043, 
                                                        364242, 364243, 364246,
                                                        364248, 700320, 700321, 
                                                        700322, 700323, 700324, 
                                                        700325, 700587, 700792, 
                                                        700793, 700794]},
    r'Background $ZZ^{*}$':     {'dids': [700600, 700601]},
    r'Signal ($m_H$ = 125 GeV)':  {'dids': [345060, 346228, 346340, 
                                            346341, 346342, 345066, 
                                            346414, 346511]},
}

# We will use the 4 leptons in the final state skim
data_samples = atom.build_dataset(defs, skim='exactly4lep', protocol='root')

### *Monte Carlo normalization*

Here we will define the MC weighting. Unlike last time, we will focus both on electrons and muons and need to account for their scale factors in the normalization.

In [ ]:
def calc_weight(data):
    weight_list = ( data["ScaleFactor_ELE"] * data["ScaleFactor_MUON"] * data["ScaleFactor_LepTRIGGER"] * 
                  data["ScaleFactor_PILEUP"] * ( data["mcWeight"] / data["sum_of_weights"]) * 
                  (data["xsec"] * data["filteff"] * data["kfac"] * lumi) )
    return weight_list

Here we will define the cuts we want to perform on our data, I will provide you with them, but feel free to tweak them and if you want, add more.

In [ ]:
# Cut on the lepton type:
# 44 = e+e-e+e-
# 48 = e+e-mu+mu-
# 52 = mu+mu-mu+mu-
def cut_lep_type(lep_type):
    sum_lep_type = ak.sum(lep_type, axis=1)
    return (sum_lep_type == 44) | (sum_lep_type == 48) | (sum_lep_type == 52)

In [ ]:
# Check to see if the total charge of the leptons is zero
def cut_lep_charge(lep_charge):
   return ak.sum(lep_charge, axis=1) == 0

In [ ]:
# Cut on the correct identification of the particle and on whether the detected leptons are well isolated
def cut_ID_Iso(el_ID, mu_ID, el_Iso, mu_Iso, pid):
    return (ak.sum(((pid == 13) & mu_ID & mu_Iso) | ((pid == 11) & el_ID & el_Iso), axis=1) == 4)

In [ ]:
# Check if either of the electron or the muon triggers fired
def cut_trig(trigE, trigM):
    return trigE | trigM

In [ ]:
# Check if at least one lepton in the event is matched with a trigger
def cut_isTrigMatched(lep_isTrigMatched):
    return ak.sum(lep_isTrigMatched, axis=1) >= 1

We also need to compute the invariant mass of the 4 leptons to see the distribution.

In [ ]:
def calc_mass(lep_pt, lep_eta, lep_phi, lep_e): 
    p4 = vector.zip({
        'pt': lep_pt,
        'eta': lep_eta,
        'phi': lep_phi,
        'E': lep_e
    })
    return ak.sum(p4, axis=1).M

Since we have multiple datasets, it is easier to define a function that parses each file, performs cuts and keeps the relevant data.

Here we will perform the cuts, compute the invariant mass, compute the weights and store everything.

In [ ]:
def process_file(filename, sample_name):
    tree = uproot.open(filename + ':analysis')
    sample_data = []

    for data in tree.iterate(filter_name=variables, library='ak', entry_stop=int(fraction*tree.num_entries), step_size=100_000):
        
        data = data[cut_lep_charge(data["lep_charge"])]
        data = data[cut_trig(data["trigE"], data["trigM"])]
        data = data[cut_isTrigMatched(data["lep_isTrigMatched"])]
        data = data[cut_ID_Iso(data["lep_isLooseID"], data["lep_isMediumID"], data["lep_isLooseIso"], data["lep_isLooseIso"], data["lep_type"])]

        data['inv_mass'] = calc_mass(data["lep_pt"], data["lep_eta"], data["lep_phi"], data["lep_e"])

        if 'Data' not in sample_name:
            data["Weight"] = calc_weight(data)
        else:
            data["Weight"] = ak.ones_like(data["met"])  # Assign weight of 1 for data samples

        sample_data.append(data)

    if sample_data is not None and len(sample_data) > 0:
        return ak.concatenate(sample_data, axis=0) 
    else:
        return ak.Array([])

We will choose the variables, define the luminosity and start processing the data files. We define the `fraction` factor to only go through a certain section of the datasets, as to not wait that long.

In [ ]:
variables = [
    "mcWeight", "ScaleFactor_LepTRIGGER", "trigE", "trigM", "lep_n", "lep_pt", 
    "lep_eta", "lep_phi", "lep_e", "lep_charge", "lep_type", "met", "met_phi", "lep_isMediumID", "lep_isLooseIso", 
    "sum_of_weights", "ScaleFactor_FTAG", "xsec", "lep_isTrigMatched", "eventNumber", "ScaleFactor_ELE", "ScaleFactor_MUON", 
    "ScaleFactor_PILEUP", "filteff", "kfac", "lep_isLooseID"
]

# Dictionary storing the data per process
data_all = {}
# Luminosity in pb^-1 You can find the luminosity of each dataset on the ATLAS Open Data Platform
lumi = 36600
# Fraction of the dataset to process
fraction = 1.0

for sample in data_samples:
    frames = []
    print(f"Processing the {sample} sample")

    for file in data_samples[sample]["list"]:
        df = process_file(file, sample)
        if df is not None and len(df) > 0:
            frames.append(df)

    data_all[sample] = ak.concatenate(frames, axis=0)

Now we can take a look at the invariant mass spectrum of the 4 leptons and see if we successfully reconstructed the $H$ mass.

In [ ]:
datasets = [
    {'data': data_all["Background $Z,t\\bar{t},t\\bar{t}+V,VVV$"]['inv_mass'], 'weights': data_all["Background $Z,t\\bar{t},t\\bar{t}+V,VVV$"]['Weight'], 'color': 'purple', 'label': r'Background $Z,t\bar{t},t\bar{t}+V,VVV$'},
    {'data': data_all["Background $ZZ^{*}$"]["inv_mass"], 'weights': data_all["Background $ZZ^{*}$"]['Weight'], 'color': 'red', 'label': r'Background $ZZ^{*}$'},
    {'data': data_all["Signal ($m_H$ = 125 GeV)"]["inv_mass"], 'weights': data_all["Signal ($m_H$ = 125 GeV)"]['Weight'], 'color': 'cyan', 'label': r'Signal ($m_H$ = 125 GeV)'},
]

fig, ax = plt.subplots()

# Use a 120-130 GeV window around the 125 GeV Higgs peak as the signal region.
signal_region_low, signal_region_high = 120.0, 130.0
bins_mass = np.arange(80, 250, 2.5)
bins_mass_centers = (bins_mass[1:] + bins_mass[:-1]) / 2

# This is for the real data points
data_x, _ = np.histogram(ak.to_numpy(data_all['Data']['inv_mass']), bins=bins_mass)
data_x_err = np.sqrt(data_x)
ax.errorbar(bins_mass_centers, data_x, yerr=data_x_err, fmt='k.', label='Data')

# This is for the MC samples
ax.hist([d['data'] for d in datasets], bins=bins_mass, weights=[d['weights'] for d in datasets], stacked=True, color=[d['color'] for d in datasets], label=[d['label'] for d in datasets])

ax.set_xlabel(r'$m_{4l}$ [GeV]')
ax.set_ylabel(f'Events / {(bins_mass[1]-bins_mass[0]):.1f} GeV')
ax.legend()
plt.show()

We can clearly see the Higgs mass peak at 125 GeV! But there is still quite a bit of background. We now introduce:

### *Machine Learning selection*

Since we record many observables for each collision, with billions nad billions of collisions, it is pretty difficult for a physicist to find the best cut on each variable, while taking into account all of the distinguishing characteristics of each process. 

Machine learning methods can combine these variables to do just that. They are able to take the datasets, learn patterns in the event kinematics, find correlations between observables and make the optimal selection for whatever our application is. These include, but are not limited to:
 - *Trigger systems and real-time filtering*: highly optimised, low-latency ML algorithms can be implemented on FPGAs and can decide whether an event is worth keeping within microseconds.
 - *Object reconstruction and identification*: classifying particle tracks, identifying electrons, photons and muons, reconstructing calorimeter clusters.
 - *Flavour tagging*: Identifying jets originating from heavy flavours ($b$, $c$ quarks) based on displaced decay vertices and impact parameters.
 - *Signal-background discrimination*: separating rare physics signals from dominating backgrounds.
And many others...

The most widely used ML models in HEP are: 
 - *Boosted Decision Trees (BDTs)*: they are the industry-standard for analyzing tabular data and are extensively used for event classification, being trained on features such as $p_T$, $η$, $ϕ$, $MET$, invariant mass.
 - *Deep Neural Networks (DNNs)*: they are mainly used for complex classification tasks, being used in modern particle identification.
 - *Generative Models (GANs, normalizing flows)*: a novel and trendy way to do fast detector simulation, replacing computationally expensive full-sim (e.g. GEANT4).

### *Decision Trees and Boosted Decision Trees (BDTs)*

To understand why BDTs are so popular in HEP, we must look how they build on simple decision trees.

#### 1. Decision Trees

A single decision tree (also known as weak learner) works by recursively splitting the feature space into regions using binary decisions, for example: Is $p_T > 45 \text{ GeV}$? Is $Δϕ < 1.2\text{ rad}$?

At each node, the algorithm evaluates every possible split across all features to maximize a metric of node purity (e.g. cross-entropy). While this process intuitive and transparent, a single decision tree is **unstable and prone to overfitting**. Fluctuations in the training dataset can drastically change the tree structure, leading to poor generalization.

#### 2. Boosted Decision Trees (BDTs)

To overcome the instability of single decision trees, we can employ **boosting**, a procedure that combines an ensemble of decision trees sequentially. Each new tree is trained specifically to correct the residual errors made by the previous sequence of trees.

A method through which this correction is made is through gradient descent optimization. When a tree fails to correctly classify a signal or background event, its residual is assigned a higher weight for the next tree to focus on.

This boosting method has some advantages when it comes to its use in HEP:
 - BDTs are very good at handling tabular data without the need for complicated data scaling and normalization.
 - They naturally perform feature selection, placing a stronger emphasis on meaningful observables.
 - They are resilient to outlier data, such as fat tails in distributions and detector artifacts.

### *Applying a BDT to $H → ZZ^* → 4ℓ$*

Now we will train a BDT classifier to distinguish simulated Higgs signal from simulated backgrounds after the preselection cuts we performed. The BDT combines reconstructed event features into a score between 0 and 1, with larger scores being more signal-like. We split the data into training, validation, and test samples. The validation sample is used to choose a score threshold, while the test sample is used to check performance.

The features we will use include:
 - The four lepton $pT$'s, ordered from highest to lowest
 - Their $η$
 - The $ΔR$ between each pair of leptons
 - The $MET$

We deliberately keep $m_{4l}$ out of the features so that we can inspect the spectrum after the selection is done.

The training gives the signal and background weights per class. For the event counts and BDT threshold choice, we will use the actual MC event weights.

In [ ]:
from itertools import combinations
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.model_selection import train_test_split

In [ ]:
def make_bdt_features(sample):
    # We want to sort the observables from the highest pT to the fourth highest over all variables for easy work
    order = ak.argsort(sample['lep_pt'], axis=1, ascending=False)
    pt = ak.to_numpy(ak.to_regular(sample['lep_pt'][order], axis=1))
    eta = ak.to_numpy(ak.to_regular(sample['lep_eta'][order], axis=1))
    phi = ak.to_numpy(ak.to_regular(sample['lep_phi'][order], axis=1))

    # We compute the difference in azimuthal angle
    delta_phi = np.abs(np.arctan2(
        np.sin(phi[:, :, None] - phi[:, None, :]),
        np.cos(phi[:, :, None] - phi[:, None, :]),
    ))
    # We compute the difference in pseudorapidity
    delta_eta = eta[:, :, None] - eta[:, None, :]
    # We do pairwise combinations of angular separation
    delta_r = np.sqrt(delta_eta**2 + delta_phi**2)
    pairs = list(combinations(range(4), 2))
    pair_delta_r = np.column_stack([delta_r[:, i, j] for i, j in pairs])

    # We take the MET as a feature as well
    met = np.asarray(ak.to_numpy(sample['met']))

    # We concatenate the features together and pass them back
    features = np.column_stack((pt, eta, pair_delta_r, met))
    return features

Now that we have a way to construct our BDT features, we do it for both signal and background samples.

In [ ]:
# Construct the signal features
signal_samples = data_all["Signal ($m_H$ = 125 GeV)"]
signal_X = make_bdt_features(signal_samples)

# Concatenate the backgrounds and construct the background features
background_samples = ak.concatenate([
    data_all['Background $Z,t\\bar{t},t\\bar{t}+V,VVV$'],
    data_all['Background $ZZ^{*}$'],
], axis=0)
background_X = make_bdt_features(background_samples)

# Construct the total features
X = np.concatenate((signal_X, background_X))

We now need to make the labels, which are the 'signatures' for each class of process, in our case signal or background. As such, we will attribute the label $1$ to signal and the label $0$ to background.

In [ ]:
y = np.concatenate((np.ones(len(signal_X)), np.zeros(len(background_X))))

We will now store the weights for after the BDT cut and the invariant masses so we can look at them later.

In [ ]:
all_weights = np.concatenate((
    np.asarray(ak.to_numpy(signal_samples['Weight'])),
    np.asarray(ak.to_numpy(background_samples['Weight'])),
))
all_masses = np.concatenate((
    np.asarray(ak.to_numpy(signal_samples['inv_mass'])),
    np.asarray(ak.to_numpy(background_samples['inv_mass'])),
))

Now we will have to do the train-validation-test split. We will split the data as follows:
 - Training: 60%
 - Validation: 20% 
 - Testing: 20%

After that, we compute the training weights per event.

In [ ]:
# We total split into train+validation and test
(X_train_val, X_test, 
 y_train_val, y_test, 
 weights_train_val, weights_test, 
 masses_train_val, masses_test) = train_test_split(X, y, all_weights, all_masses,
                                                   test_size=0.2,
                                                   stratify=y)

# We split train+validation into train and validation
(X_train, X_val,
 y_train, y_val,
 weights_train, weights_val,
 masses_train, masses_val) = train_test_split(X_train_val, y_train_val, weights_train_val, masses_train_val,
                                              test_size=0.25, 
                                              stratify=y_train_val)

# We compute the total number of events for each class
n_signal_train = np.count_nonzero(y_train == 1)
n_background_train = np.count_nonzero(y_train == 0)
# We compute per event weights
train_weights = np.where(
    y_train == 1,
    0.5 / n_signal_train,
    0.5 / n_background_train,
)

Now, we can finally define our classifier and train it!

In [ ]:
bdt = GradientBoostingClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=2,
    random_state=42,
)
bdt.fit(X_train, y_train, sample_weight=train_weights)

We now apply our trained BDT to the validation sample in order to optimize the BDT threshold. For each value, we compute $S/\sqrt{B}$ and choose the threshold which maximizes this significance. This is just a teaching optimization and should not be taken as a full statistical significance!

In [ ]:
# Score the validation events with the trained BDT
val_scores = bdt.predict_proba(X_val)[:, 1]
threshold_grid = np.linspace(0.0, 1.0, 51)
# Rescale the validation MC weights since we did not use the entire dataset
val_class_fractions = {
    label: np.count_nonzero(y_val == label) / np.count_nonzero(y == label)
    for label in np.unique(y)
}
val_weights = weights_val / np.asarray([
    val_class_fractions[label] for label in y_val
])

validation_signal_yields = []
validation_background_yields = []
# Estimate S and B at every score cut using the rescaled MC weights
for threshold in threshold_grid:
    passed = val_scores >= threshold
    # We take the signal and background counts from the validation data that pass the discriminant
    signal_yield = np.sum(val_weights[(y_val == 1) & passed])
    background_yield = np.sum(val_weights[(y_val == 0) & passed])
    validation_signal_yields.append(signal_yield)
    validation_background_yields.append(background_yield)
validation_signal_yields = np.asarray(validation_signal_yields)
validation_background_yields = np.asarray(validation_background_yields)
# We compute the S/sqrt(B) for the validation set
validation_significances = np.divide(
    validation_signal_yields,
    np.sqrt(validation_background_yields),
    out=np.full_like(validation_signal_yields, np.nan, dtype=float),
    where=(validation_signal_yields > 0) & (validation_background_yields > 0),
)

# Optimize the cut on the validation dataset
bdt_threshold = float(threshold_grid[np.nanargmax(validation_significances)])
print(f'Optimized BDT threshold: {bdt_threshold:.2f}')

After the threshold is fixed using validation, evaluate it once on the independent test sample. We also take a look at some performance metrics for the BDT, such as ROC AUC, and the most important features in the classification. We can also apply the mask to the actual data (*not MC!*) to see how many events from the data pass as signal.

In [ ]:
# Evaluate the chosen model on the test sample
test_scores = bdt.predict_proba(X_test)[:, 1]
# We print the AUC 
print(f'Test ROC AUC (unweighted): {roc_auc_score(y_test, test_scores):.3f}')
# We take a look at the most important features in training
print('Most important features:', sorted(zip(
    ['pT1', 'pT2', 'pT3', 'pT4', 'eta1', 'eta2', 'eta3', 'eta4',
     'dR12', 'dR13', 'dR14', 'dR23', 'dR24', 'dR34', 'MET'],
    bdt.feature_importances_,
), key=lambda item: item[1], reverse=True)[:8])

# We can also plot the ROC and take a look at it
test_fpr, test_tpr, _ = roc_curve(y_test, test_scores)
test_auc = roc_auc_score(y_test, test_scores)
fig, ax_roc = plt.subplots()
ax_roc.plot(test_fpr, test_tpr, label=f'Test ROC (AUC = {test_auc:.3f})')
ax_roc.plot([0, 1], [0, 1], 'k--', label='Random classifier')
ax_roc.set_xlabel('Background acceptance')
ax_roc.set_ylabel('Signal efficiency')
ax_roc.legend()
plt.show()

# We apply the selection to the data and see how many events pass as signal
data_sample = data_all['Data']
data_x_bdt = make_bdt_features(data_sample)
data_scores_bdt = bdt.predict_proba(data_x_bdt)[:, 1]
print(f'Data events passing BDT: {np.count_nonzero(data_scores_bdt >= bdt_threshold)} / {len(data_scores_bdt)}')

Now we want to take a look at how good the signal-background separation is at each BDT output. For that we will have to see how many events we correctly classified and misclassified as either signal or background.

In [ ]:
# We will comptue the number of true/false positives/negatives
test_truth_signal = y_test == 1
test_pred_signal = test_scores >= bdt_threshold

true_signal_selected = test_truth_signal & test_pred_signal
background_misclassified = ~test_truth_signal & test_pred_signal
signal_rejected = test_truth_signal & ~test_pred_signal
background_rejected = ~test_truth_signal & ~test_pred_signal

# Rescale the test weights to the full-sample
test_class_fractions = {
    label: np.count_nonzero(y_test == label) / np.count_nonzero(y == label)
    for label in np.unique(y)
}
test_weights = weights_test / np.asarray([
    test_class_fractions[label] for label in y_test
])
selected_signal_yield = np.sum(test_weights[true_signal_selected])
selected_background_yield = np.sum(test_weights[background_misclassified])
selected_s_over_b = (
    selected_signal_yield / selected_background_yield
    if selected_background_yield > 0 else np.nan
)

# We can now take a look at classification efficiency
n_true_signal = np.count_nonzero(test_truth_signal)
n_true_background = np.count_nonzero(~test_truth_signal)
signal_efficiency = np.count_nonzero(true_signal_selected) / n_true_signal
background_rejection = np.count_nonzero(background_rejected) / n_true_background
print(f'Test decisions at score >= {bdt_threshold:.2f}:')
print(f'Signal selected (True Positive rate): {np.count_nonzero(true_signal_selected)} / {n_true_signal} = {(np.count_nonzero(true_signal_selected) / n_true_signal):.2f}')
print(f'Signal rejected (False Negative rate): {np.count_nonzero(signal_rejected)} / {n_true_signal} = {(np.count_nonzero(signal_rejected) / n_true_signal):.2f}')
print(f'Background falsely selected (False Positive rate): {np.count_nonzero(background_misclassified)} / {n_true_background} = {(np.count_nonzero(background_misclassified) / n_true_background):.2f}')
print(f'Background rejected (True negative rate): {np.count_nonzero(background_rejected)} / {n_true_background} = {(np.count_nonzero(background_rejected) / n_true_background):.2f}')
print(f'Signal efficiency: {signal_efficiency:.1%}')
print(f'Background rejection: {background_rejection:.1%}')

# Compare the shapes and look at the decision boundary
score_bins = np.linspace(0, 1, 51)
fig, ax = plt.subplots()
ax.hist(test_scores[test_truth_signal], bins=score_bins, histtype='step', label='True signal', density=True)
ax.hist(test_scores[~test_truth_signal], bins=score_bins, histtype='step', label='True background', density=True)
ax.axvline(bdt_threshold, color='black', linestyle='--', label=f'Optimal BDT threshold = {bdt_threshold:.2f}')
ax.set_xlabel('BDT output')
ax.set_ylabel('Test event counts')
ax.legend()
fig.tight_layout()
plt.show()

The invariant mass was calculated before the BDT and stored as `inv_mass`. We do not need to reconstruct it again: apply the BDT score selection to the stored mass values.

In [ ]:
# Reuse the validation-selected threshold for the independent test mass spectrum.
test_pass = test_scores >= bdt_threshold
test_signal = (y_test == 1) & test_pass
test_background = (y_test == 0) & test_pass
test_class_fractions = {
    label: np.count_nonzero(y_test == label) / np.count_nonzero(y == label)
    for label in np.unique(y)
}
test_weights = weights_test / np.asarray([
    test_class_fractions[label] for label in y_test
])
test_masses = masses_test

# Count selected test MC in the mass window and sum its scaled weights for expected yields.
test_in_signal_region = (test_masses >= signal_region_low) & (test_masses < signal_region_high)
test_signal_in_region = test_signal & test_in_signal_region
test_background_in_region = test_background & test_in_signal_region
# Apply the same BDT cut to data before counting its observed signal-region events.
data_pass = data_scores_bdt >= bdt_threshold
data_masses_all = np.asarray(ak.to_numpy(data_sample['inv_mass']))
data_in_signal_region_after_bdt = data_pass & (data_masses_all >= signal_region_low) & (data_masses_all < signal_region_high)
print(f'After BDT, signal region {signal_region_low:.0f} <= m4l < {signal_region_high:.0f} GeV:')
print(f'  Data observed: {np.count_nonzero(data_in_signal_region_after_bdt)} events')
print(f'  Signal MC: {np.count_nonzero(test_signal_in_region)} test events; weighted yield = {np.sum(test_weights[test_signal_in_region]):.2f}')
print(f'  Background MC: {np.count_nonzero(test_background_in_region)} test events; weighted yield = {np.sum(test_weights[test_background_in_region]):.2f}')

# Apply the same fixed threshold to data; data has no truth labels for BDT training.
data_masses_bdt = np.asarray(ak.to_numpy(data_sample['inv_mass']))[data_pass]
bins_bdt = np.arange(80, 250, 2.5)
bin_centers_bdt = (bins_bdt[1:] + bins_bdt[:-1]) / 2
data_counts_bdt, _ = np.histogram(data_masses_bdt, bins=bins_bdt)

fig, ax = plt.subplots()
ax.hist(
    [test_masses[test_background], test_masses[test_signal]],
    bins=bins_bdt,
    weights=[test_weights[test_background], test_weights[test_signal]],
    stacked=True,
    label=['Background', 'Higgs signal'],
    color=['red', 'cyan'],
)
ax.errorbar(
    bin_centers_bdt,
    data_counts_bdt,
    yerr=np.sqrt(data_counts_bdt),
    fmt='k.',
    label='Data after BDT selection',
)
ax.set_xlabel(r'$m_{4\ell}$ [GeV]')
ax.set_ylabel(f'Events / {(bins_bdt[1] - bins_bdt[0]):.1f} GeV')
ax.legend()
plt.show()

### *What to do now?*

Here are some ideas to consider and test:
 - Play with the pre-selection cuts: you can tweak them, add more or even get rid of them completely and implement them directly in the BDT!
 - Modify the hyperparameters of the BDT: change the number of estimators, depth and learning rate and see how the results change! Note: if you increase the number of estimators and depth, expect much longer training times.
 - Do some statistics: compute $S/B$, $S/\sqrt{B}$ and other metrics in order to see how your analysis performs.
- Try to fit the mass of the Higgs boson.